# RAG-Powered Document Assistant

## 2.1 Load & Inspect

This notebook implements a Retrieval-Augmented Generation (RAG) system
for question answering over the complete Sherlock Holmes canon by
Arthur Conan Doyle.

The RAG pipeline includes:

1. Document preparation and inspection
2. Text extraction
3. Chunking
4. Embedding generation
5. Vector storage
6. Retrieval
7. Prompt construction
8. Answer generation
9. Evaluation
10. Exporting the vector store and configuration

In [4]:
import sys
print(sys.executable)

C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\.venv\Scripts\python.exe


In [5]:
from pathlib import Path
from pypdf import PdfReader

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Source PDF
PDF_PATH = PROJECT_ROOT / "data" / "cano.pdf"

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())
print("PDF size (MB):", round(PDF_PATH.stat().st_size / (1024 * 1024), 2))

Project root: C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project
PDF path: C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\data\cano.pdf
PDF exists: True
PDF size (MB): 4.82


In [6]:
reader = PdfReader(PDF_PATH)

num_pages = len(reader.pages)

print("Number of pages:", num_pages)

Number of pages: 987


In [8]:
pages = []
failed_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    try:
        text = page.extract_text()

        if text and text.strip():
            pages.append({
                "page_number": page_number,
                "text": text.strip()
            })
        else:
            failed_pages.append(page_number)

    except Exception as e:
        failed_pages.append(page_number)
        print(f"Failed to parse page {page_number}: {e}")

print("Total PDF pages:", num_pages)
print("Pages with extracted text:", len(pages))
print("Pages without usable text:", len(failed_pages))

Total PDF pages: 987
Pages with extracted text: 883
Pages without usable text: 104


In [9]:
print("First extracted page:")
print(pages[0]["text"][:2000])

print("\n" + "=" * 80)

print("Last extracted page:")
print(pages[-1]["text"][:2000])

First extracted page:
The Complete Sherlock Holmes
Arthur Conan Doyle

Last extracted page:
The Adventure of the Retired Colourman
“Write a message.”
“Exactly. You would like to tell people how you
died. No use writing on paper. That would be seen.
If you wrote on the wall someone might rest upon it.
Now, look here! Just above the skirting is scribbled
with a purple indelible pencil: ‘We we—’ That’s all.”
“What do you make of that?”
“Well, it’s only a foot above the ground. The poor
devil was on the ﬂoor dying when he wrote it. He
lost his senses before he could ﬁnish.”
“He was writing, ‘We were murdered.’ ”
“That’s how I read it. If you ﬁnd an indelible
pencil on the body—”
“We’ll look out for it, you may be sure. But those
securities? Clearly there was no robbery at all. And
yet he did possess those bonds. We veriﬁed that.”
“You may be sure he has them hidden in a safe
place. When the whole elopement had passed into
history, he would suddenly discover them and an-
nounce that the gui

In [10]:
print("First 20 pages without usable text:")
print(failed_pages[:20])

First 20 pages without usable text:
[6, 8, 10, 12, 40, 42, 70, 72, 124, 126, 128, 140, 142, 154, 156, 166, 178, 180, 190, 192]


In [11]:
page_lengths = [len(page["text"]) for page in pages]

print("Total extracted characters:", sum(page_lengths))
print("Average characters per page:", round(sum(page_lengths) / len(page_lengths)))
print("Shortest extracted page:", min(page_lengths))
print("Longest extracted page:", max(page_lengths))

Total extracted characters: 3584289
Average characters per page: 4059
Shortest extracted page: 12
Longest extracted page: 5415


In [12]:
import pandas as pd

pages_df = pd.DataFrame(pages)

pages_df.head()

,page_number,text
0,1,The Complete Sherlock Holmes\nArthur Conan Doyle
1,2,This text is provided to you “as-is” without a...
2,3,T able of contents\nA Study In Scarlet . . . ....
3,4,The Return of Sherlock Holmes\nThe Adventure o...
4,5,The Case-Book of Sherlock Holmes\nPreface . . ...


### Text Extraction Findings

The PDF contains 987 pages in total. Text extraction was successful for
883 pages, while 104 pages did not contain usable extracted text.

A total of 3,584,289 characters were extracted, with an average of
approximately 4,059 characters per extracted page.

The extracted text was manually inspected on the first and last
available pages and was found to contain readable book content.

The pages without usable text may correspond to illustrations,
blank/formatting pages, or pages whose content is not represented as
extractable text. Since this project focuses on text-based question
answering, the extracted textual content will be used as the main RAG
corpus.

## 2.2 Chunking Strategy

The extracted pages are divided into smaller overlapping text chunks
to improve retrieval quality.

A Recursive Character Text Splitter is used because it attempts to
split text at natural boundaries such as paragraphs and sentences
before falling back to smaller separators.

Configuration:

- Chunk size: 1000 characters
- Chunk overlap: 150 characters

The overlap helps preserve context when an important sentence or piece
of information lies near a chunk boundary.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [14]:
chunks = []

for page in pages:
    page_chunks = text_splitter.split_text(page["text"])

    for chunk_index, chunk_text in enumerate(page_chunks):
        chunks.append({
            "text": chunk_text,
            "page_number": page["page_number"],
            "chunk_index": chunk_index,
            "source": "The Complete Sherlock Holmes"
        })

print("Total chunks:", len(chunks))

Total chunks: 4519


In [15]:
print("First chunk:")
print(chunks[0]["text"])

print("\n" + "=" * 80)

print("First chunk metadata:")
print({
    "page_number": chunks[0]["page_number"],
    "chunk_index": chunks[0]["chunk_index"],
    "source": chunks[0]["source"]
})

First chunk:
The Complete Sherlock Holmes
Arthur Conan Doyle

First chunk metadata:
{'page_number': 1, 'chunk_index': 0, 'source': 'The Complete Sherlock Holmes'}


In [16]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\New\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [17]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Batches:   0%|          | 0/142 [00:00<?, ?it/s]

Number of embeddings: 4519
Embedding dimension: 384


In [18]:
import chromadb
from pathlib import Path

# Directory where the vector database will be persisted
VECTOR_STORE_PATH = PROJECT_ROOT / "data" / "vector_store"

VECTOR_STORE_PATH.mkdir(parents=True, exist_ok=True)

# Create persistent Chroma client
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_PATH)
)

# Create collection
collection = chroma_client.get_or_create_collection(
    name="sherlock_holmes"
)

print("Vector store path:", VECTOR_STORE_PATH)
print("Collection name:", collection.name)

Vector store path: C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\data\vector_store
Collection name: sherlock_holmes


In [19]:
batch_size = 500

for start in range(0, len(chunks), batch_size):
    end = min(start + batch_size, len(chunks))

    batch_chunks = chunks[start:end]
    batch_embeddings = embeddings[start:end]

    collection.add(
        ids=[f"chunk_{i}" for i in range(start, end)],
        documents=[chunk["text"] for chunk in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        metadatas=[
            {
                "source": chunk["source"],
                "page_number": chunk["page_number"],
                "chunk_index": chunk["chunk_index"]
            }
            for chunk in batch_chunks
        ]
    )

    print(f"Stored chunks {start} → {end - 1}")

print("\nTotal documents in Chroma:", collection.count())

Stored chunks 0 → 499
Stored chunks 500 → 999
Stored chunks 1000 → 1499
Stored chunks 1500 → 1999
Stored chunks 2000 → 2499
Stored chunks 2500 → 2999
Stored chunks 3000 → 3499
Stored chunks 3500 → 3999
Stored chunks 4000 → 4499
Stored chunks 4500 → 4518

Total documents in Chroma: 4519


## 2.4 Retrieval & Prompting

The retrieval stage searches the Chroma vector store for the most
semantically relevant chunks to a user question.

The question is embedded using the same embedding model used for the
document chunks. Chroma then retrieves the nearest chunks based on
vector similarity.

In [20]:
def retrieve_documents(query, top_k=5):
    query_embedding = embedding_model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):
        retrieved_chunks.append({
            "text": results["documents"][0][i],
            "page_number": results["metadatas"][0][i]["page_number"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks

In [21]:
query = "Who is Dr. Watson?"

retrieved = retrieve_documents(query, top_k=5)

for i, result in enumerate(retrieved, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Page:", result["page_number"])
    print("Distance:", round(result["distance"], 4))
    print(result["text"][:1000])


--- Retrieved Chunk 1 ---
Page: 940
Distance: 0.8233
“As you like, Mr. Holmes. You will, I am sure,
understand my having some reserves in the matter.”
“You will appreciate it, Watson, when I tell you
that this gentleman, Mr. Trevor Bennett, is profes-
sional assistant to the great scientist, lives under his
roof, and is engaged to his only daughter. Certainly
we must agree that the professor has every claim upon
his loyalty and devotion. But it may best be shown
by taking the necessary steps to clear up this strange
mystery.”
“I hope so, Mr. Holmes. That is my one object.
Does Dr. Watson know the situation?”
“I have not had time to explain it.”
“Then perhaps I had better go over the ground
again before explaining some fresh developments.”
“I will do so myself,” said Holmes, “in order to
show that I have the events in their due order. The
professor, Watson, is a man of European reputation.
His life has been academic. There has never been a
breath of scandal. He is a widower with one da

### Prompt Construction

The retrieved chunks are combined with the user's question to create
a grounded prompt for the language model.

The model is instructed to answer using the provided context and to
avoid introducing information that is not supported by the retrieved
content.

In [22]:
def build_prompt(question, retrieved_chunks):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Source {i} | Page {chunk['page_number']}]\n"
            f"{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a question-answering assistant for The Complete Sherlock Holmes.

Answer the user's question using only the provided context.

If the answer cannot be determined from the context, say:
"I could not find enough information in the provided context."

Always mention the relevant source page(s) in your answer.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [23]:
question = "Who is Dr. Watson?"

prompt = build_prompt(question, retrieved)

print(prompt[:5000])

You are a question-answering assistant for The Complete Sherlock Holmes.

Answer the user's question using only the provided context.

If the answer cannot be determined from the context, say:
"I could not find enough information in the provided context."

Always mention the relevant source page(s) in your answer.

Context:
[Source 1 | Page 940]
“As you like, Mr. Holmes. You will, I am sure,
understand my having some reserves in the matter.”
“You will appreciate it, Watson, when I tell you
that this gentleman, Mr. Trevor Bennett, is profes-
sional assistant to the great scientist, lives under his
roof, and is engaged to his only daughter. Certainly
we must agree that the professor has every claim upon
his loyalty and devotion. But it may best be shown
by taking the necessary steps to clear up this strange
mystery.”
“I hope so, Mr. Holmes. That is my one object.
Does Dr. Watson know the situation?”
“I have not had time to explain it.”
“Then perhaps I had better go over the ground
again 

In [24]:
from ollama import chat

response = chat(
    model="qwen3:1.7b",
    messages=[
        {
            "role": "user",
            "content": "Who is Sherlock Holmes?"
        }
    ]
)

print(response["message"]["content"])

Sherlock Holmes is a fictional detective created by **Arthur Conan Doyle** in the 1880s. He is one of the most iconic figures in detective fiction and is often regarded as one of the greatest detectives in literature. Here's a concise overview:

### Key Traits:
- **Analytical and Logical**: Holmes is renowned for his exceptional deductive reasoning, sharp mind, and ability to solve complex mysteries.
- **Witty and Sarcastic**: His dialogue is sharp, witty, and often laced with humor.
- **Unconventional**: He challenges traditional detective work by relying on logic, observation, and sometimes unconventional methods.

### Notable Works:
- **"The Adventure of the Blanched Soldier"** (1887): A classic story showcasing his deductive skills.
- **"The Adventure of the Dancing Men"** (1891): A story where Holmes uses a unique pattern to solve a mystery.
- **"The Adventure of the Murders at the Old House"** (1892): A pivotal tale involving a haunted house and a murder.

### Legacy:
- Holmes is

In [25]:
question = "Who is Dr. Watson?"

retrieved = retrieve_documents(question, top_k=5)

prompt = build_prompt(question, retrieved)

response = chat(
    model="qwen3:1.7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print(answer)

Dr. Watson is the companion and friend of Sherlock Holmes, as mentioned in the provided context. He is described as a man of European reputation, a widower, and a key figure in Holmes's investigations. The context references his role in discussing cases and his relationship with Holmes, particularly in the context of solving mysteries. 

Relevant source:  
[Source 1 | Page 940]  
[Source 2 | Page 809]


In [26]:
def ask_rag(question, top_k=5):
    # Retrieve relevant chunks
    retrieved_chunks = retrieve_documents(question, top_k=top_k)

    # Build grounded prompt
    prompt = build_prompt(question, retrieved_chunks)

    # Generate answer using Ollama
    response = chat(
        model="qwen3:1.7b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    return answer, retrieved_chunks

In [27]:
answer, sources = ask_rag("Who is Dr. Watson?")

print(answer)

print("\nRetrieved Sources:")
for source in sources:
    print(
        f"- Page {source['page_number']} "
        f"(distance: {source['distance']:.4f})"
    )

Dr. Watson is the narrator of *The Complete Sherlock Holmes* series, a close companion and friend of Sherlock Holmes. He is mentioned in the context as a key figure in Holmes's investigations, often interacting with him and providing background details. 

Relevant sources:  
- **Source 2 | Page 809**: Watson is described as a friend of Holmes and someone he has worked with.  
- **Source 3 | Page 33**: Watson is referred to as "Doctor Watson" in a conversation involving Holmes.  
- **Source 4 | Page 77**: Watson is explicitly called the "very man" by the narrator, indicating his role as a trusted companion.  

Thus, Dr. Watson is the narrator and a central figure in the stories, often collaborating with Holmes.

Retrieved Sources:
- Page 940 (distance: 0.8233)
- Page 809 (distance: 0.8585)
- Page 33 (distance: 0.8726)
- Page 77 (distance: 0.8879)
- Page 143 (distance: 0.8914)


## 2.6 Evaluation

The RAG system is evaluated using a set of test questions covering
different aspects of the Sherlock Holmes corpus.

For each question, we examine:

- Whether the retrieved chunks are relevant
- Whether the generated answer is grounded in the retrieved context
- Whether the answer is factually correct
- The source pages used to support the answer
- Failure cases and possible improvements

In [28]:
evaluation_questions = [
    "Who is Dr. Watson?",
    "Who is Sherlock Holmes?",
    "Where does Sherlock Holmes live?",
    "Who is Mrs. Hudson?",
    "What is Sherlock Holmes's profession?",
    "What is Dr. Watson's profession?",
    "Who is Inspector Lestrade?",
    "What is the relationship between Holmes and Watson?",
    "What kind of cases does Sherlock Holmes investigate?",
    "Who is Professor Moriarty?"
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 10


In [29]:
evaluation_results = []

for question in evaluation_questions:
    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    
    answer, sources = ask_rag(question, top_k=5)
    
    source_pages = [source["page_number"] for source in sources]
    
    evaluation_results.append({
        "question": question,
        "answer": answer,
        "source_pages": source_pages
    })
    
    print("\nAnswer:")
    print(answer)
    
    print("\nRetrieved pages:")
    print(source_pages)


Question: Who is Dr. Watson?

Answer:
Dr. Watson is a companion and colleague of Sherlock Holmes in the context provided. He is mentioned as a fellow investigator and a close friend, often referred to in conversations between Holmes and others. The context does not specify his profession beyond his role as a friend and collaborator, and there is no detailed information about his background or other roles. 

Relevant source: [Source 2 | Page 809] mentions him as a friend of the professor, and [Source 3 | Page 33] and [Source 5 | Page 143] refer to him as a colleague and companion of Holmes. 

I could not find enough information in the provided context.

Retrieved pages:
[940, 809, 33, 77, 143]

Question: Who is Sherlock Holmes?

Answer:
Sherlock Holmes is a detective created by Arthur Conan Doyle, as stated in **Source 4 | Page 1** of *The Complete Sherlock Holmes*. He is portrayed as a brilliant, eccentric detective renowned for his sharp intellect and deductive skills. The context do

In [30]:
for result in evaluation_results:
    print("\n" + "=" * 100)
    print("QUESTION:", result["question"])
    print("\nANSWER:")
    print(result["answer"])
    
    print("\nRETRIEVED CONTEXT:")
    for i, source in enumerate(
        retrieve_documents(result["question"], top_k=5),
        start=1
    ):
        print(f"\n--- Chunk {i} | Page {source['page_number']} ---")
        print(source["text"][:1500])


QUESTION: Who is Dr. Watson?

ANSWER:
Dr. Watson is a companion and colleague of Sherlock Holmes in the context provided. He is mentioned as a fellow investigator and a close friend, often referred to in conversations between Holmes and others. The context does not specify his profession beyond his role as a friend and collaborator, and there is no detailed information about his background or other roles. 

Relevant source: [Source 2 | Page 809] mentions him as a friend of the professor, and [Source 3 | Page 33] and [Source 5 | Page 143] refer to him as a colleague and companion of Holmes. 

I could not find enough information in the provided context.

RETRIEVED CONTEXT:

--- Chunk 1 | Page 940 ---
“As you like, Mr. Holmes. You will, I am sure,
understand my having some reserves in the matter.”
“You will appreciate it, Watson, when I tell you
that this gentleman, Mr. Trevor Bennett, is profes-
sional assistant to the great scientist, lives under his
roof, and is engaged to his only da

### Evaluation Results

The system was tested using 10 questions covering character identity,
relationships, professions, locations, and types of cases.

Each result was reviewed based on retrieval relevance, answer
grounding, and factual correctness.

The evaluation also revealed several failure cases where the retriever
returned weak or insufficient context. In such cases, the language
model sometimes refused to answer, while in other cases it generated
information that was not directly supported by the retrieved chunks.

In [31]:
evaluation_labels = [
    {
        "question": "Who is Dr. Watson?",
        "retrieval": "Partial",
        "grounded": True,
        "correct": "Partial",
        "notes": "Retrieved relevant mentions, but no clear profession/background."
    },
    {
        "question": "Who is Sherlock Holmes?",
        "retrieval": "Weak",
        "grounded": False,
        "correct": "Partial",
        "notes": "Retrieved mostly book titles; answer included unsupported general knowledge."
    },
    {
        "question": "Where does Sherlock Holmes live?",
        "retrieval": "Strong",
        "grounded": True,
        "correct": True,
        "notes": "Page 745 directly states that Holmes lives on a small farm near Eastbourne."
    },
    {
        "question": "Who is Mrs. Hudson?",
        "retrieval": "Partial",
        "grounded": True,
        "correct": "Partial",
        "notes": "Retrieved a mention of Mrs. Hudson but not a direct description of her role."
    },
    {
        "question": "What is Sherlock Holmes's profession?",
        "retrieval": "Weak",
        "grounded": True,
        "correct": False,
        "notes": "Retrieved context did not explicitly provide his profession."
    },
    {
        "question": "What is Dr. Watson's profession?",
        "retrieval": "Weak",
        "grounded": True,
        "correct": False,
        "notes": "Retrieved context mentioned 'Doctor Watson' but did not directly explain his profession."
    },
    {
        "question": "Who is Inspector Lestrade?",
        "retrieval": "Strong",
        "grounded": True,
        "correct": True,
        "notes": "Retrieved chunks directly describe Lestrade and his involvement in investigations."
    },
    {
        "question": "What is the relationship between Holmes and Watson?",
        "retrieval": "Strong",
        "grounded": True,
        "correct": True,
        "notes": "Retrieved chunks directly support their friendship and collaboration."
    },
    {
        "question": "What kind of cases does Sherlock Holmes investigate?",
        "retrieval": "Strong",
        "grounded": True,
        "correct": True,
        "notes": "Retrieved chunks discuss difficult crimes and investigations."
    },
    {
        "question": "Who is Professor Moriarty?",
        "retrieval": "Strong",
        "grounded": True,
        "correct": True,
        "notes": "Retrieved chunks directly describe Moriarty's criminal activities and intellect."
    }
]

In [32]:
import pandas as pd

evaluation_df = pd.DataFrame(evaluation_labels)

evaluation_df

,question,retrieval,grounded,correct,notes
0,Who is Dr. Watson?,Partial,True,Partial,"Retrieved relevant mentions, but no clear prof..."
1,Who is Sherlock Holmes?,Weak,False,Partial,Retrieved mostly book titles; answer included ...
2,Where does Sherlock Holmes live?,Strong,True,True,Page 745 directly states that Holmes lives on ...
3,Who is Mrs. Hudson?,Partial,True,Partial,Retrieved a mention of Mrs. Hudson but not a d...
4,What is Sherlock Holmes's profession?,Weak,True,False,Retrieved context did not explicitly provide h...
5,What is Dr. Watson's profession?,Weak,True,False,Retrieved context mentioned 'Doctor Watson' bu...
6,Who is Inspector Lestrade?,Strong,True,True,Retrieved chunks directly describe Lestrade an...
7,What is the relationship between Holmes and Wa...,Strong,True,True,Retrieved chunks directly support their friend...
8,What kind of cases does Sherlock Holmes invest...,Strong,True,True,Retrieved chunks discuss difficult crimes and ...
9,Who is Professor Moriarty?,Strong,True,True,Retrieved chunks directly describe Moriarty's ...


In [33]:
question = "What is Sherlock Holmes's profession?"

retrieved = retrieve_documents(question, top_k=10)

for i, chunk in enumerate(retrieved, start=1):
    print(f"\n{'=' * 70}")
    print(f"Rank {i} | Page {chunk['page_number']} | Distance: {chunk['distance']:.4f}")
    print(chunk["text"])


Rank 1 | Page 123 | Distance: 0.5755
The Adventures of Sherlock Holmes

Rank 2 | Page 745 | Distance: 0.6006
Preface
Preface
The friends of Mr. Sherlock Holmes will be glad to learn that he is still alive and well, though
somewhat crippled by occasional attacks of rheumatism. He has, for many years, lived in a small
farm upon the downs ﬁve miles from Eastbourne, where his time is divided between philosophy
and agriculture. During this period of rest he has refused the most princely offers to take up
various cases, having determined that his retirement was a permanent one. The approach of the
German war caused him, however, to lay his remarkable combination of intellectual and practical
activity at the disposal of the government, with historical results which are recounted in His Last
Bow. Several previous experiences which have lain long in my portfolio have been added to His
Last Bow so as to complete the volume.
John H. Watson, M. D.
739

Rank 3 | Page 853 | Distance: 0.6171
The Cas

In [34]:
question = "What does Sherlock Holmes do for a living?"

retrieved = retrieve_documents(question, top_k=10)

for i, chunk in enumerate(retrieved, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| Distance: {chunk['distance']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 123 | Distance: 0.5746
The Adventures of Sherlock Holmes

Rank 2 | Page 745 | Distance: 0.6094
Preface
Preface
The friends of Mr. Sherlock Holmes will be glad to learn that he is still alive and well, though
somewhat crippled by occasional attacks of rheumatism. He has, for many years, lived in a small
farm upon the downs ﬁve miles from Eastbourne, where his time is divided between philosophy
and agriculture. During this period of rest he has refused the most princely offers to take up
various cases, having determined that his retirement was a permanent one. The approach of the
German war caused him, however, to lay his remarkable combination of intellectual and practical
activity at the disposal of the government, with historical results which are recounted in His Last
Bow. Several previous experiences which have lain long in my portfolio have been added to His
Last Bow so as to complete the volume.
John H. Watson, M. D.
739

Rank 3 | Page 419 | Distance: 0.6632
The Ret

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

chunk_texts = [chunk["text"] for chunk in chunks]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4519, 21891)


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

def keyword_retrieve(query, top_k=10):
    query_vector = tfidf_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:
        results.append({
            "text": chunks[index]["text"],
            "page_number": chunks[index]["page_number"],
            "chunk_index": chunks[index]["chunk_index"],
            "source": chunks[index]["source"],
            "score": similarities[index]
        })

    return results

In [37]:
question = "What is Sherlock Holmes's profession?"

keyword_results = keyword_retrieve(question, top_k=10)

for i, chunk in enumerate(keyword_results, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| TF-IDF Score: {chunk['score']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 419 | TF-IDF Score: 0.3578
The Return of Sherlock Holmes

Rank 2 | Page 853 | TF-IDF Score: 0.3219
The Case-Book of Sherlock Holmes

Rank 3 | Page 123 | TF-IDF Score: 0.3030
The Adventures of Sherlock Holmes

Rank 4 | Page 283 | TF-IDF Score: 0.2719
The Memoirs of Sherlock Holmes

Rank 5 | Page 843 | TF-IDF Score: 0.2017
His Last Bow
An Epilogue of Sherlock Holmes

Rank 6 | Page 150 | TF-IDF Score: 0.1532
turn, we never know where to ﬁnd the man himself.
He’ll crack a crib in Scotland one week, and be raising
money to build an orphanage in Cornwall the next.
I’ve been on his track for years and have never set
eyes on him yet.”
“I hope that I may have the pleasure of introduc-
ing you to-night. I’ve had one or two little turns also
with Mr. John Clay, and I agree with you that he is at
the head of his profession. It is past ten, however, and
quite time that we started. If you two will take the
ﬁrst hansom, Watson and I will follow in the second.”
Sherlock Holmes was not v

In [38]:
def hybrid_retrieve(query, top_k=5, candidate_k=20):
    semantic_results = retrieve_documents(query, top_k=candidate_k)
    keyword_results = keyword_retrieve(query, top_k=candidate_k)

    scores = {}
    chunk_data = {}

    # Semantic ranking
    for rank, result in enumerate(semantic_results, start=1):
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        scores[key] = scores.get(key, 0) + 1 / (60 + rank)
        chunk_data[key] = result

    # Keyword ranking
    for rank, result in enumerate(keyword_results, start=1):
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        scores[key] = scores.get(key, 0) + 1 / (60 + rank)
        chunk_data[key] = result

    # Sort by combined score
    ranked_keys = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    results = []

    for key in ranked_keys[:top_k]:
        result = chunk_data[key].copy()
        result["hybrid_score"] = scores[key]
        results.append(result)

    return results

In [39]:
question = "What is Sherlock Holmes's profession?"

hybrid_results = hybrid_retrieve(
    question,
    top_k=10,
    candidate_k=20
)

for i, chunk in enumerate(hybrid_results, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| Hybrid Score: {chunk['hybrid_score']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 123 | Hybrid Score: 0.0323
The Adventures of Sherlock Holmes

Rank 2 | Page 419 | Hybrid Score: 0.0320
The Return of Sherlock Holmes

Rank 3 | Page 853 | Hybrid Score: 0.0320
The Case-Book of Sherlock Holmes

Rank 4 | Page 283 | Hybrid Score: 0.0310
The Memoirs of Sherlock Holmes

Rank 5 | Page 1 | Hybrid Score: 0.0294
The Complete Sherlock Holmes
Arthur Conan Doyle

Rank 6 | Page 745 | Hybrid Score: 0.0161
Preface
Preface
The friends of Mr. Sherlock Holmes will be glad to learn that he is still alive and well, though
somewhat crippled by occasional attacks of rheumatism. He has, for many years, lived in a small
farm upon the downs ﬁve miles from Eastbourne, where his time is divided between philosophy
and agriculture. During this period of rest he has refused the most princely offers to take up
various cases, having determined that his retirement was a permanent one. The approach of the
German war caused him, however, to lay his remarkable combination of intellectual an

In [40]:
def is_useful_chunk(text, min_words=20):
    return len(text.split()) >= min_words

In [41]:
useful_chunks = [
    chunk for chunk in chunks
    if is_useful_chunk(chunk["text"])
]

print("Original chunks:", len(chunks))
print("Useful chunks:", len(useful_chunks))
print("Removed chunks:", len(chunks) - len(useful_chunks))

Original chunks: 4519
Useful chunks: 4448
Removed chunks: 71


In [42]:
useful_texts = [chunk["text"] for chunk in useful_chunks]

tfidf_vectorizer_filtered = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

tfidf_matrix_filtered = tfidf_vectorizer_filtered.fit_transform(
    useful_texts
)

print("Filtered TF-IDF matrix shape:", tfidf_matrix_filtered.shape)

Filtered TF-IDF matrix shape: (4448, 21889)


In [43]:
def keyword_retrieve_filtered(query, top_k=10):
    query_vector = tfidf_vectorizer_filtered.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix_filtered
    ).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:
        results.append({
            "text": useful_chunks[index]["text"],
            "page_number": useful_chunks[index]["page_number"],
            "chunk_index": useful_chunks[index]["chunk_index"],
            "source": useful_chunks[index]["source"],
            "score": similarities[index]
        })

    return results

In [44]:
question = "What is Sherlock Holmes's profession?"

filtered_results = keyword_retrieve_filtered(
    question,
    top_k=10
)

for i, chunk in enumerate(filtered_results, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| TF-IDF Score: {chunk['score']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 150 | TF-IDF Score: 0.1533
turn, we never know where to ﬁnd the man himself.
He’ll crack a crib in Scotland one week, and be raising
money to build an orphanage in Cornwall the next.
I’ve been on his track for years and have never set
eyes on him yet.”
“I hope that I may have the pleasure of introduc-
ing you to-night. I’ve had one or two little turns also
with Mr. John Clay, and I agree with you that he is at
the head of his profession. It is past ten, however, and
quite time that we started. If you two will take the
ﬁrst hansom, Watson and I will follow in the second.”
Sherlock Holmes was not very communicative dur-
ing the long drive and lay back in the cab humming
the tunes which he had heard in the afternoon. We
rattled through an endless labyrinth of gas-lit streets
until we emerged into Farrington Street.
“We are close there now,” my friend remarked.
“This fellow Merryweather is a bank director, and
personally interested in the matter. I thought it as

Rank 2 | Pa

In [45]:
def hybrid_retrieve_filtered(query, top_k=5, candidate_k=20):
    semantic_results = retrieve_documents(query, top_k=candidate_k)
    keyword_results = keyword_retrieve_filtered(query, top_k=candidate_k)

    # Remove very short chunks from semantic results
    semantic_results = [
        result
        for result in semantic_results
        if is_useful_chunk(result["text"])
    ]

    scores = {}
    chunk_data = {}

    # Semantic ranking
    for rank, result in enumerate(semantic_results, start=1):
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        scores[key] = scores.get(key, 0) + 1 / (60 + rank)
        chunk_data[key] = result

    # Keyword ranking
    for rank, result in enumerate(keyword_results, start=1):
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        scores[key] = scores.get(key, 0) + 1 / (60 + rank)
        chunk_data[key] = result

    # Sort by combined score
    ranked_keys = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    results = []

    for key in ranked_keys[:top_k]:
        result = chunk_data[key].copy()
        result["hybrid_score"] = scores[key]
        results.append(result)

    return results

In [46]:
question = "What is Sherlock Holmes's profession?"

hybrid_results = hybrid_retrieve_filtered(
    question,
    top_k=10,
    candidate_k=20
)

for i, chunk in enumerate(hybrid_results, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| Hybrid Score: {chunk['hybrid_score']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 745 | Hybrid Score: 0.0164
Preface
Preface
The friends of Mr. Sherlock Holmes will be glad to learn that he is still alive and well, though
somewhat crippled by occasional attacks of rheumatism. He has, for many years, lived in a small
farm upon the downs ﬁve miles from Eastbourne, where his time is divided between philosophy
and agriculture. During this period of rest he has refused the most princely offers to take up
various cases, having determined that his retirement was a permanent one. The approach of the
German war caused him, however, to lay his remarkable combination of intellectual and practical
activity at the disposal of the government, with historical results which are recounted in His Last
Bow. Several previous experiences which have lain long in my portfolio have been added to His
Last Bow so as to complete the volume.
John H. Watson, M. D.
739

Rank 2 | Page 150 | Hybrid Score: 0.0164
turn, we never know where to ﬁnd the man himself.
He’ll crack a crib in

In [47]:
import numpy as np

def weighted_hybrid_retrieve(query, top_k=5, candidate_k=20):
    # Get candidates from both retrievers
    semantic_results = retrieve_documents(query, top_k=candidate_k)
    keyword_results = keyword_retrieve_filtered(query, top_k=candidate_k)

    # Store candidates by page + chunk
    candidates = {}

    for result in semantic_results:
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        if is_useful_chunk(result["text"]):
            candidates[key] = {
                "text": result["text"],
                "page_number": result["page_number"],
                "chunk_index": result["chunk_index"],
                "source": result["source"],
                "semantic_score": 1 - result["distance"],
                "keyword_score": 0.0
            }

    for result in keyword_results:
        key = (
            result["page_number"],
            result["chunk_index"]
        )

        if key not in candidates:
            candidates[key] = {
                "text": result["text"],
                "page_number": result["page_number"],
                "chunk_index": result["chunk_index"],
                "source": result["source"],
                "semantic_score": 0.0,
                "keyword_score": result["score"]
            }
        else:
            candidates[key]["keyword_score"] = result["score"]

    # Convert candidates to list
    results = list(candidates.values())

    # Normalize scores
    semantic_scores = np.array(
        [r["semantic_score"] for r in results]
    )

    keyword_scores = np.array(
        [r["keyword_score"] for r in results]
    )

    def min_max_normalize(scores):
        min_score = scores.min()
        max_score = scores.max()

        if max_score == min_score:
            return np.zeros_like(scores)

        return (scores - min_score) / (max_score - min_score)

    semantic_normalized = min_max_normalize(semantic_scores)
    keyword_normalized = min_max_normalize(keyword_scores)

    # Weighted combination
    for i, result in enumerate(results):
        result["hybrid_score"] = (
            0.4 * semantic_normalized[i]
            + 0.6 * keyword_normalized[i]
        )

    # Sort by final score
    results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return results[:top_k]

In [48]:
question = "What is Sherlock Holmes's profession?"

weighted_results = weighted_hybrid_retrieve(
    question,
    top_k=10,
    candidate_k=20
)

for i, chunk in enumerate(weighted_results, start=1):
    print(f"\n{'=' * 70}")
    print(
        f"Rank {i} | Page {chunk['page_number']} "
        f"| Hybrid Score: {chunk['hybrid_score']:.4f}"
    )
    print(chunk["text"])


Rank 1 | Page 150 | Hybrid Score: 0.6000
turn, we never know where to ﬁnd the man himself.
He’ll crack a crib in Scotland one week, and be raising
money to build an orphanage in Cornwall the next.
I’ve been on his track for years and have never set
eyes on him yet.”
“I hope that I may have the pleasure of introduc-
ing you to-night. I’ve had one or two little turns also
with Mr. John Clay, and I agree with you that he is at
the head of his profession. It is past ten, however, and
quite time that we started. If you two will take the
ﬁrst hansom, Watson and I will follow in the second.”
Sherlock Holmes was not very communicative dur-
ing the long drive and lay back in the cab humming
the tunes which he had heard in the afternoon. We
rattled through an endless labyrinth of gas-lit streets
until we emerged into Farrington Street.
“We are close there now,” my friend remarked.
“This fellow Merryweather is a bank director, and
personally interested in the matter. I thought it as

Rank 2 | Pa

### Retrieval Improvement

The initial semantic retrieval did not always return the most relevant chunks for questions containing specific terms such as "profession".

To improve retrieval, a hybrid approach was tested by combining semantic similarity with TF-IDF keyword matching.

Short and low-information chunks were also filtered out before keyword retrieval. This reduced noise from title-only or very short chunks and improved the ranking of relevant textual evidence.

For the question "What is Sherlock Holmes's profession?", the hybrid retriever successfully retrieved relevant passages, including passages on pages 554, 324, and 20 that explicitly discuss Holmes's profession, calling, and detective work.

The hybrid approach will therefore be used as the final retrieval strategy for the RAG pipeline.


In [49]:
def ask_rag(question, top_k=5):
    retrieved_chunks = hybrid_retrieve_filtered(
        question,
        top_k=top_k
    )

    prompt = build_prompt(question, retrieved_chunks)

    response = chat(
        model="qwen3:1.7b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    return answer, retrieved_chunks

In [50]:
answer, sources = ask_rag(
    "What is Sherlock Holmes's profession?"
)

print(answer)

print("\nRetrieved Sources:")
for source in sources:
    print(
        f"- Page {source['page_number']} "
        f"(hybrid score: {source['hybrid_score']:.4f})"
    )

The context indicates that Sherlock Holmes is a **detective** based on the preface in Source 1 (Page 745), which states he has "refused the most princely offers to take up various cases, having determined that his retirement was a permanent one." While the preface mentions his retirement and pursuits in philosophy and agriculture, the explicit mention of his profession as a **detective** is derived from the story's context (Source 2, Page 150), where he is described as a figure who has been on Watson's track but never set eyes on him. 

**Answer:**  
Sherlock Holmes's profession is a **detective**, as indicated in the preface (Source 1) and the story's context (Source 2).

Retrieved Sources:
- Page 745 (hybrid score: 0.0164)
- Page 150 (hybrid score: 0.0164)
- Page 855 (hybrid score: 0.0161)
- Page 554 (hybrid score: 0.0161)
- Page 234 (hybrid score: 0.0159)


### Retrieval and Generation Result

The hybrid retrieval strategy improved the retrieval of relevant evidence for the question about Sherlock Holmes's profession.

The retrieved context included passages that explicitly refer to Holmes's profession, calling, and detective work. The generated answer correctly identified Sherlock Holmes as a detective.

However, the generated response also included an interpretation that was not directly supported by every cited page. This demonstrates that retrieval quality and citation grounding should both be considered during evaluation.

Overall, the hybrid retriever provided sufficiently relevant context for the RAG system and will be used for the final evaluation.


## 2.7 Export

The Chroma vector store has been persisted to disk so that the embeddings do not need to be regenerated when the application starts.

The RAG configuration is also exported to a JSON file. This configuration records the chunking parameters, embedding model, vector store, and retrieval strategy used to build the system.


In [51]:
import json

config = {
    "chunk_size": 1000,
    "chunk_overlap": 150,
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store": "ChromaDB",
    "collection_name": "sherlock_holmes",
    "retrieval_strategy": "hybrid",
    "semantic_weight": 0.4,
    "keyword_weight": 0.6,
    "top_k": 5
}

CONFIG_PATH = PROJECT_ROOT / "data" / "rag_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Configuration saved to:")
print(CONFIG_PATH)

Configuration saved to:
C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\data\rag_config.json


In [52]:
print("Vector store exists:", VECTOR_STORE_PATH.exists())
print("Vector store path:", VECTOR_STORE_PATH)

print("\nConfiguration exists:", CONFIG_PATH.exists())
print("Configuration path:", CONFIG_PATH)

Vector store exists: True
Vector store path: C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\data\vector_store

Configuration exists: True
Configuration path: C:\Users\New\OneDrive - Faculty Of Science (Ain Shams University)\RAG_Assistant_Project\data\rag_config.json
